# Sentiment Analysis with Transformer Encoder

Let's build a BERT-style Transformer encoder for binary sentiment classification.

**We will learn:**
- How encoder-only Transformers work
- The difference between bidirectional and causal attention
- Using the CLS token for classification
- Visualizing attention weights

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import matplotlib.pyplot as plt
import math
import string

## 1. Dataset: Movie Reviews

We create a small dataset of movie reviews with binary labels.

Label **1** means positive review, Label **0** means negative review.

In [ ]:
# Positive reviews
positive_reviews = [
    "I absolutely loved this movie! The acting was superb and the storyline kept me engaged from start to finish. The cinematography was breathtaking and the soundtrack perfectly complemented every scene.",
    "Brilliant film with an excellent plot. The characters were well developed and the ending was perfect. I would highly recommend this to anyone who enjoys quality cinema.",
    "One of the best movies I have seen this year. Great cinematography and outstanding performances from the entire cast. The director did an amazing job bringing this story to life.",
    "Fantastic movie that exceeded all my expectations. The special effects were stunning and the story was both touching and thrilling. I cannot wait to watch it again.",
    "A masterpiece of modern cinema. Every scene was beautifully crafted and the emotional depth of the characters was remarkable. This film will stay with me for a long time.",
    "Absolutely wonderful from beginning to end. The performances were outstanding and the script was intelligent and engaging. A must-watch for any film enthusiast.",
    "Incredible storytelling combined with superb acting. The pacing was perfect and I was completely absorbed in the narrative. One of the finest films in recent years.",
    "Loved every single minute of this film. The attention to detail was impressive and the character development was exceptional. Highly entertaining and deeply moving.",
    "A triumph in every aspect. The direction was flawless, the cinematography was gorgeous, and the performances were powerful. This is cinema at its finest.",
    "Beautifully made film with a compelling story. The actors delivered phenomenal performances and the visual effects were spectacular. I was thoroughly impressed throughout.",
    "Outstanding movie that delivers on all fronts. The plot twists kept me guessing and the emotional moments were genuinely touching. A remarkable achievement in filmmaking.",
    "Exceptional film with brilliant performances. The screenplay was clever and the direction was inspired. This is the kind of movie that reminds you why you love cinema.",
    "Thoroughly enjoyed this masterpiece. The storytelling was captivating and the production values were top-notch. Every element came together perfectly to create something special.",
    "Magnificent film that showcases the best of cinema. The performances were riveting and the story was both entertaining and thought-provoking. Absolutely loved it.",
    "A stunning achievement in filmmaking. The visual style was breathtaking and the narrative was compelling from start to finish. This movie sets a new standard for excellence."
]

# Negative reviews
negative_reviews = [
    "This was a complete waste of time and money. The plot made no sense and the acting was wooden at best. I found myself checking my watch every few minutes hoping it would end soon.",
    "Completely boring and poorly executed. I struggled to stay awake during the entire film. The script was terrible and the performances were unconvincing throughout.",
    "Worst movie experience ever. Bad script, bad acting, and terrible direction throughout. I cannot believe I sat through the entire thing. Save yourself the disappointment.",
    "Absolutely dreadful from start to finish. The story was confusing and the characters were one-dimensional. The special effects looked cheap and the dialogue was cringe-worthy.",
    "A complete disaster of a film. Nothing worked here - not the plot, not the acting, not the direction. I have rarely been so disappointed by a movie.",
    "Painfully bad in every possible way. The pacing was terrible, the story made no sense, and the performances were embarrassing. I wanted to leave halfway through.",
    "Utterly disappointing and poorly made. The plot holes were enormous and the acting was amateurish. This film was a waste of talented actors and a good premise.",
    "Terrible movie that fails on every level. The script was nonsensical, the direction was uninspired, and the whole thing felt like a chore to sit through.",
    "One of the worst films I have ever seen. The story was predictable and boring, the acting was stiff, and the production quality was surprisingly poor.",
    "Completely unwatchable garbage. The dialogue was awful, the plot was ridiculous, and the performances were wooden. I regret every minute I spent watching this.",
    "Dreadful film with no redeeming qualities. The pacing dragged, the characters were unlikeable, and the story went nowhere. A total waste of time and money.",
    "Horribly executed disaster. The special effects were laughable, the acting was terrible, and the plot made absolutely no sense. I cannot recommend avoiding this enough.",
    "Abysmal movie that insults the intelligence of its audience. The writing was lazy, the direction was poor, and the whole experience was deeply unsatisfying.",
    "Shockingly bad film. The story was incoherent, the performances were flat, and the technical aspects were subpar. This is two hours of my life I will never get back.",
    "Monumentally disappointing waste of potential. Everything that could go wrong did go wrong. The execution was poor and the final product was unwatchable."
]

# Create dataset with labels
reviews = [(text, 1) for text in positive_reviews] + [(text, 0) for text in negative_reviews]

print(f"Total reviews: {len(reviews)}")
print(f"Positive: {len(positive_reviews)}, Negative: {len(negative_reviews)}")

In [ ]:
reviews

## 2. Building Vocabulary

We create a vocabulary from all reviews and add special tokens.
The CLS token will be used for classification.

In [ ]:
def build_vocabulary(reviews):
    vocab = {"<PAD>": 0, "<CLS>": 1, "<UNK>": 2}

    for text, _ in reviews:
        # Remove punctuation and convert to lowercase
        text = text.translate(str.maketrans('', '', string.punctuation)).lower()
        
        for word in text.split():
            if word not in vocab:
                vocab[word] = len(vocab)
    return vocab

vocabulary = build_vocabulary(reviews)
inverse_vocab = {idx: word for word, idx in vocabulary.items()}
vocab_size = len(vocabulary)

print(f"Vocabulary size: {vocab_size}")

## 3. Text Preprocessing

Convert reviews to token indices. We add the CLS token at the beginning of each review.

In [ ]:
def encode_review(text, vocab, max_length=60):
    # Remove punctuation and convert to lowercase
    text = text.translate(str.maketrans('', '', string.punctuation)).lower()
    
    # Add CLS token at the beginning, use <UNK> for unknown words
    tokens = [vocab["<CLS>"]] + [vocab.get(word, vocab["<UNK>"]) for word in text.split()]
    
    # Truncate if too long
    if len(tokens) > max_length:
        tokens = tokens[:max_length]
    
    # Pad if too short. 
    while len(tokens) < max_length:
        tokens.append(vocab["<PAD>"])
    
    return torch.tensor(tokens, dtype=torch.long)

# Prepare dataset
max_seq_length = 60
encoded_reviews = []
labels = []

for text, label in reviews:
    encoded = encode_review(text, vocabulary, max_seq_length)
    encoded_reviews.append(encoded)
    labels.append(label)

print(f"Encoded {len(encoded_reviews)} reviews")
print(f"Sequence length: {max_seq_length}")

In [ ]:
encoded_reviews[0]

## 4. Positional Encoding

Adds position information to word embeddings using sine and cosine functions. (Sinusoidal Positional Encoding)

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, embedding_dim, max_length=100):
        super().__init__()
        position_encoding = torch.zeros(max_length, embedding_dim)
        position = torch.arange(0, max_length).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0, embedding_dim, 2).float() * (-math.log(10000.0) / embedding_dim))
        
        position_encoding[:, 0::2] = torch.sin(position * div_term)
        position_encoding[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('position_encoding', position_encoding.unsqueeze(0))
    
    def forward(self, x):
        return x + self.position_encoding[:, :x.size(1)]

## 5. Sentiment Classifier Model

Use PyTorch's built-in TransformerEncoder for simplicity.

The model has three main parts:
1. Word embeddings with positional encoding
2. Transformer encoder layers
3. Classification head

In [ ]:
class SentimentTransformer(nn.Module):
    def __init__(self, vocab_size, embedding_dim=64, num_heads=4, num_layers=2, feedforward_dim=128, num_classes=2, dropout=0.1):
        super().__init__()
        
        # Word embeddings
        self.word_embedding = nn.Embedding(vocab_size, embedding_dim)
        self.positional_encoding = PositionalEncoding(embedding_dim)
        
        # Transformer encoder using PyTorch built-in
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embedding_dim,
            nhead=num_heads,
            dim_feedforward=feedforward_dim,
            dropout=dropout,
            batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        # Classification head
        self.classifier = nn.Linear(embedding_dim, num_classes)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x):
        # Create padding mask
        padding_mask = (x == 0) # Used to mask out padding tokens in self-attention. 
        
        # Embeddings with positional encoding
        x = self.dropout(self.positional_encoding(self.word_embedding(x)))
        
        # Pass through transformer encoder
        x = self.transformer_encoder(x, src_key_padding_mask=padding_mask)
        
        # Use CLS token (first token) for classification
        cls_representation = x[:, 0, :]
        logits = self.classifier(cls_representation)
        
        return logits

## 6. Training the Model

Train the model using cross-entropy loss and track both loss and accuracy.

In [ ]:
# Initialize model
model = SentimentTransformer(
    vocab_size=vocab_size,
    embedding_dim=64, # embedding dimension
    num_heads=4, # number of attention heads
    num_layers=2, # number of encoder layers
    feedforward_dim=128, # feedforward dimension
    num_classes=2, # number of classes
    dropout=0.1 # dropout - regularization
)

optimizer = optim.Adam(model.parameters(), lr=0.0003)
criterion = nn.CrossEntropyLoss()

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# Training loop
num_epochs = 300
losses = []
accuracies = []

for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for review, label in zip(encoded_reviews, labels):
        review = review.unsqueeze(0)  # Add batch dimension
        label_tensor = torch.tensor([label], dtype=torch.long)
        
        optimizer.zero_grad()
        outputs = model(review)
        loss = criterion(outputs, label_tensor)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
        # Calculate accuracy
        predicted = outputs.argmax(dim=1)
        correct += (predicted == label_tensor).sum().item()
        total += 1
    
    avg_loss = total_loss / len(encoded_reviews)
    accuracy = 100 * correct / total
    losses.append(avg_loss)
    accuracies.append(accuracy)
    
    if (epoch + 1) % 50 == 0:
        print(f"Epoch {epoch+1}/{num_epochs}, Loss: {avg_loss:.4f}, Accuracy: {accuracy:.2f}%")

In [ ]:
# Plot results

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(losses)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training Loss')
ax1.grid(True)

ax2.plot(accuracies)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.set_title('Training Accuracy')
ax2.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# Save model

torch.save(model.state_dict(), 'sentiment_transformer.pth')
print("\nModel saved!")

## 7. Testing the Model

Test on some examples to see how well the model classifies sentiment.

In [ ]:
def predict_sentiment(text, model, vocab, max_length=60):
    model.eval()
    encoded = encode_review(text, vocab, max_length).unsqueeze(0)
    
    with torch.no_grad():
        outputs = model(encoded)
        probabilities = F.softmax(outputs, dim=1)
        predicted_class = outputs.argmax(dim=1).item() # 0 or 1
    
    sentiment = "POSITIVE" if predicted_class == 1 else "NEGATIVE"
    confidence = probabilities[0][predicted_class].item() * 100
    
    return sentiment, confidence

In [ ]:
# Test examples
# Test some review with OUT OF VOCABULARY - OOV token and test it!

test_reviews = [
    "I loved every moment of it the movie but it was too long.",
    "Terrible movie. Complete waste of my time and money.",
    "The acting was superb and the story was captivating throughout.",
    "Boring and poorly made. I regret watching this.",
    "A masterpiece of cinema with brilliant performances.",
    "The plot was predictable and the acting was wooden.",
    "Wow what a House"
]

print("SENTIMENT PREDICTIONS")
for review in test_reviews:
    sentiment, confidence = predict_sentiment(review, model, vocabulary, max_seq_length)
    print(f" Review: {review}")
    print(f"Prediction: {sentiment} (Confidence: {confidence:.1f}%)\n")


## 8. Understanding the Architecture

**Encoder-Only vs Encoder-Decoder:**

| Feature | Sentiment (Encoder-Only) | Translation (Encoder-Decoder) |
|---------|-------------------------|-------------------------------|
| Architecture | Encoder only | Encoder plus Decoder |
| Attention Type | Bidirectional | Encoder: bidirectional, Decoder: causal |
| Task | Classification | Sequence-to-sequence |
| Output | Single label | Variable length sequence |
| Special Token | CLS | SOS, EOS |

**Why Bidirectional Attention Works Here:**
- Classification needs full context understanding
- No need to prevent seeing future words
- CLS token aggregates information from entire sequence

**The CLS Token:**
- Special token added at the beginning
- Learns to aggregate sequence information
- Used by BERT and similar models
- Its representation is used for classification

## Objectives and Playground:

1. **TASK 1**: Model Debugging: Analyzing why certain sentences fail (overfitting/vocabulary check).
2. **TASK 2**: Testing the model without Positional Encoding.

3. Load a sentiment dataset and implement a HuggingFace Tokenizer to convert raw text into input IDs, attention masks, and token types.
4. Load a pre-trained `AutoModelForSequenceClassification` (e.g., BERT or DistilBERT) and configure the classification head for binary or multi-class sentiment tasks.
5. Assess the model's performance using metrics like accuracy, precision, recall, and F1-score.